In [41]:
import urllib.parse
from argparse import ArgumentParser

import torch
from pytorch_lightning import LightningModule, Trainer, LightningDataModule
from pytorch_lightning.loggers import TensorBoardLogger 
from torch import nn
from torch.nn import functional as F

import numpy as np 
import glob
import os
import random
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import multiprocessing
import math
import matplotlib.pyplot as plt
from scipy import interpolate

import pytorch_lightning as pl
from torch.autograd import Variable


from sklearn.metrics import f1_score
from pl_bolts.callbacks import PrintTableMetricsCallback
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import KFold
import warnings

torch.backends.cudnn.enabled=False
warnings.filterwarnings("ignore")

In [42]:
from sklearn.metrics import cohen_kappa_score
from imblearn.metrics import sensitivity_score
from imblearn.metrics import specificity_score

In [43]:
FOLD = 1               #[0, 1, 2, 3, ...... 9]
NUM_FOLDS = 10
NUM_LAYERS = 3

WINDOW = 9
STRIDE = 4
BATCH_SIZE = 128

EPOCH = 60
EMB_DIM = 512  #512
NUM_CLASS = 5
HIDDEN_DIM = 100
DROPOUT = 0
LR = 0.001

CNN_CH =1
CNN_LAYERS = [2, 2, 2, 2]


DATASET = 'iSLEEPS'
CHANNEL = '1ch_EEG_'
MODEL = 'SEresnet18_3LSTM_'



PROJECT = "ISBI_"
GROUP = DATASET+CHANNEL+MODEL+"_W"+str(WINDOW)+"_S"+str(STRIDE)+"_Training_First_15"

In [44]:
def getData(all_subjects):
    # this func returns a list of all the 
    # windowed subjects present for train/test/val
    
    all_list = []
    for folder in all_subjects:
        f= glob.glob(os.path.join(folder,"*.npz"))
        subs = [ i for i in f]
        all_list.extend(subs)
    return all_list

In [45]:
#getting all the subjects present after original pre-processing
all_path_npz = 'iSLEEPS/saswata/sleep_project_data/nimhans_preprocessed_data/numpy_subjects'
data_dir = glob.glob(os.path.join(all_path_npz,"*.npz"))
all_subjects = [ i for i in data_dir]
all_subjects.sort()
len(all_subjects)

100

In [46]:
# getting all the subjects (all_subjects_folders)
# availiable for train/val/test/split
base_folder = f'iSLEEPS/saswata/sleep_project_data/nimhans_preprocessed_data/numpy_samples'
data_dir = glob.glob(os.path.join(base_folder,"*"))
all_subjects_folders = [ i for i in data_dir]
all_subjects_folders.sort()
len(all_subjects_folders)

100

In [47]:
# Use all subjects for testing instead of cross-validation split
testing_index = np.arange(len(all_subjects_folders))  # All indices from 0 to 99
asf = np.array(all_subjects_folders)
test_subjects = asf  # All subjects

print("Using all subjects for testing")
print("Total Test Subjects: ", len(test_subjects))
print("Testing indices: ", testing_index)

# Get all windowed data for testing
TEST_windowed_list = getData(test_subjects)
print("Total Test Windowed Instance: ", len(TEST_windowed_list))
print('--------------')
print(testing_index)


Using all subjects for testing
Total Test Subjects:  100
Testing indices:  [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95
 96 97 98 99]
Total Test Windowed Instance:  23192
--------------
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71
 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95
 96 97 98 99]


In [48]:
class CustomDataset(Dataset):
    def __init__(self, subjects_list):
        self.subjects_list_paths = subjects_list
        self.channel = 2
   
    def __len__(self):
        return len(self.subjects_list_paths)

    def __getitem__(self, idx):
        file = np.load(self.subjects_list_paths[idx])
    
        return file['x'][:,:,self.channel].astype(np.float32), np.squeeze(file['y'][math.ceil(file['y'].shape[0]/2)], axis=-1)

In [49]:
class DataModule(LightningDataModule):
    def __init__(self,train_folder,val_folder, batch_size):#,test_folder):
        super().__init__()
        self.train_sub_list = train_folder
        self.val_sub_list = val_folder
#         self.test_sub_list = test_folder
        self.batch_size = batch_size
        
    def train_dataloader(self):
        trainDataset = CustomDataset(self.train_sub_list)
        trainDataloader = DataLoader(trainDataset, batch_size=self.batch_size, drop_last=True)
        return trainDataloader

    def val_dataloader(self):
        valDataset = CustomDataset(self.val_sub_list)
        valDataloader = DataLoader(valDataset, batch_size=self.batch_size, drop_last=True)
        return valDataloader

#     def test_dataloader(self):
        # testDataset = CustomDataset(self.test_sub_list)
#         testDataloader = DataLoader(testDataset, batch_size=self.BATCH_SIZE, drop_last=True)
#         return testDataloader

In [50]:
class SELayer(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SELayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),    #down
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),    #up
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

In [51]:
def conv3x1(in_planes, out_planes, stride=1):
    """3x3 convolution with padding"""
    return nn.Conv1d(in_planes, out_planes, kernel_size=7, stride=stride,
                     padding=3, bias=False)

def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv1d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None,is_last = False):
        super(BasicBlock, self).__init__()
        self.is_last = is_last
        self.conv1 = conv3x1(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)

        self.relu2 = nn.ReLU(inplace=True)
        self.conv2 = conv3x1(planes, planes)
        self.bn2 = nn.BatchNorm1d(planes)
        self.se = SELayer(planes)
        self.downsample = downsample
        self.stride = stride
        self.dropout = nn.Dropout(.2)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)
        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        preact = out
        
        out = self.relu2(out)

        if self.is_last:
            return out, preact
        else:
            return out


In [52]:
class ResNet(pl.LightningModule):

    def __init__(self, block, layers, in_channel=1, out_channel=5, zero_init_residual=True):
        super(ResNet, self).__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv1d(in_channel, 64, kernel_size=15, stride=2, padding=7,
                               bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(5, 10)
        self.fc = nn.Linear(512 * block.expansion, out_channel)
        self.sof = nn.Softmax(dim=1)

        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        # Zero-initialize the last BN in each residual branch,
        # so that the residual branch starts with zeros, and each residual block behaves like an identity.
        # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

        """
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                n = m.kernel_size[0] * m.kernel_size[0] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm1d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
        """
                          
    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm1d(planes * block.expansion),
            )

        layers = []
        
        layers.append(block(self.inplanes, planes, stride, downsample,is_last=(blocks == 1)))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            
            layers.append(block(self.inplanes, planes,is_last=(i == blocks-1)))

        return nn.Sequential(*layers)



    def forward(self, x):
#         print(x.shape)
#         x = x.permute(0,2,1)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x, f2_pre = self.layer1(x)
        x, f3_pre = self.layer2(x)
        x, f4_pre = self.layer3(x)
        x, f5_pre = self.layer4(x)
#         print(x.shape)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
     
#         x = self.fc(x)
#         x = self.sof(x)     

        return x

In [53]:
def resnet18(pretrained=False, cnn_layers = None, in_lead = 5, **kwargs):
    """Constructs a ResNet-18 model., ag, is_feat=False, preact=False
    Args:
        pretrained (bool): If True, returns a model pre-trained on ImageNet
    """
    model = ResNet(BasicBlock, cnn_layers,in_channel = in_lead, **kwargs)
    return model

In [54]:
class SSModel(pl.LightningModule):
    def __init__(self, embedding_dim, hidden_dim, num_layers, num_class, cnn, drop_prob=0.5):
        super(SSModel, self).__init__()
        self.num_class = num_class
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim
        self.cnn = cnn
        
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, dropout=drop_prob, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(drop_prob)
        self.fc1 = nn.Linear(2*hidden_dim, 2000)
        self.fc2 = nn.Linear(2000, 200)
        self.fc3 = nn.Linear(200, self.num_class)
        self.softmax = nn.LogSoftmax(dim = -1)
        
        
    def forward(self, x, hidden):
        batch_size = x.size(0)
        newX = torch.zeros(x.shape[0],x.shape[1],512).to(x)  #24064
        #print(next(self.cnn.parameters()).device)
        for i in range(x.size(1)):
            l = self.cnn(torch.unsqueeze(x[:,i,:], 1))
            newX[:,i,:] = l
        
        lstm_out, hidden = self.lstm(newX, hidden) #embeds

        mid = lstm_out[:,math.ceil(lstm_out.shape[1]/2),:]
        final_state = mid.contiguous().view(-1, 2*self.hidden_dim)
        out = self.dropout(final_state)
        out = self.fc1(out)#6000->2000
        out = self.fc2(out)#2000->200
        out = self.fc3(out)#200->5
        out = self.softmax(out)

        return out, hidden
    
    
#     def init_hidden(self, batch_size):
#         weight = next(self.parameters()).data
#         hidden = (weight.new(2*self.num_layers, batch_size, self.hidden_dim).zero_(),
#                       weight.new(2*self.num_layers, batch_size, self.hidden_dim).zero_())
#         return hidden
    

In [55]:
class Classifier(pl.LightningModule):
    def __init__(self, lstm_model,lr,hidden_dim):
        super().__init__()
        self.lstm_model = lstm_model
        self.lr = lr
        self.hidden_dim = hidden_dim
        self.h_0 = torch.zeros(2, 100, self.hidden_dim)
        self.c_0 = torch.zeros(2, 100, self.hidden_dim)
        self.h = (self.h_0,self.c_0)
        
        # save hyper-parameters to self.hparams (auto-logged by W&B)
#         self.save_hyperparameters()
        
    def step(self, batch, batch_idx):
        inputs, labels  = batch
        
        self.h_0 = torch.zeros(6, inputs.shape[0], self.hidden_dim).to(inputs) #CHANGE STACKS LSTM HERE
        self.c_0 = torch.zeros(6, inputs.shape[0], self.hidden_dim).to(inputs)

        self.h = (self.h_0,self.c_0)
#         print('------------>>>>')
        out_lstm,self.h = self.lstm_model(inputs,self.h)
#         print('------------>>>>')
    
        criterion = nn.NLLLoss()
        
#         print(out_lstm.shape)
#         print("Labels",labels.shape)
        loss = criterion(out_lstm,labels.type(torch.cuda.LongTensor))
#         print(out_lstm.dtype)
#         print(out_lstm.shape)

#             print(inputs.dtype, labels.dtype, out_lstm.dtype)
#             print(count, out_lstm,labels)
        return loss, {"loss": loss}, out_lstm ,labels


    def training_step(self, batch, batch_idx):

        loss, logs,out,true = self.step(batch, batch_idx)
        self.log_dict({f"train_{k}": v for k, v in logs.items()}, on_step=True, on_epoch=False, prog_bar=True)
        return loss #{'loss': loss}

    def validation_step(self, batch, batch_idx):
        loss, logs,out,true  = self.step(batch, batch_idx)
        self.log_dict({f"val_{k}": v for k, v in logs.items()}, prog_bar=True)
        
        return (loss, out, true)
    
    def validation_epoch_end(self, val_out_tup):
        # outs is a list of whatever you returned in `validation_step`
        
        f_loss = []
        f_preds = []
        f_truth = []
        
        for loss, pred, tgt in val_out_tup:
            l = loss.detach().cpu().numpy()
            p = pred.detach().cpu().numpy()
            t = tgt.detach().cpu().numpy()

            f_loss.append(l)
            f_preds.extend(p)
            f_truth.extend(t)
            
        f_loss = np.array(f_loss)
        f_preds = np.array(f_preds)
        f_truth = np.array(f_truth)

        Floss = f_loss.mean()
        y_preds = np.argmax(f_preds,axis=-1)
        y_truth = f_truth #ground truth

        f1_all = f1_score(y_truth, y_preds, average='macro')
        
        kappa = cohen_kappa_score(y_truth, y_preds)
        sensitivity = sensitivity_score(y_truth, y_preds, average='macro')
        specificity = specificity_score(y_truth, y_preds, average='macro')
        
        f1_classWise = f1_score(y_truth, y_preds, average=None)
        acc_all = accuracy_score(y_truth, y_preds)
        
        self.log("vLoss", Floss, sync_dist=True, on_epoch=True)
        self.log("vAcc", acc_all, sync_dist=True, on_epoch=True)
        self.log("Kappa", kappa, sync_dist=True, on_epoch=True)
        self.log("sensitivity", sensitivity, sync_dist=True, on_epoch=True)
        self.log("specificity", specificity, sync_dist=True, on_epoch=True)
        self.log("vF1", f1_all, sync_dist=True, on_epoch=True, prog_bar=True)
        self.log("vClassF1_W", f1_classWise[0], sync_dist=True, on_epoch=True)
        self.log("vClassF1_N1", f1_classWise[1], sync_dist=True, on_epoch=True)
        self.log("vClassF1_N2", f1_classWise[2], sync_dist=True, on_epoch=True)
        self.log("vClassF1_N3", f1_classWise[3], sync_dist=True, on_epoch=True)
        self.log("vClassF1_REM", f1_classWise[4], sync_dist=True, on_epoch=True)

        print("  -x- ")


    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)



In [ ]:
# Load the best model and evaluate on testing patients
def load_model_and_evaluate(best_model_path, test_subjects_folders):
    # Recreate the model architecture first
    modelCNN = resnet18(cnn_layers=CNN_LAYERS, in_lead=1)
    lstm_model = SSModel(EMB_DIM, HIDDEN_DIM, NUM_LAYERS, NUM_CLASS, modelCNN, DROPOUT)
    model_classifier = Classifier(lstm_model, lr=LR, hidden_dim=HIDDEN_DIM)
    
    # Load the trained weights
    checkpoint = torch.load(best_model_path, map_location='cpu')
    model_classifier.load_state_dict(checkpoint['state_dict'])
    model_classifier.eval()
    
    # Get test data
    TEST_windowed_list = getData(test_subjects_folders)
    test_dataset = CustomDataset(TEST_windowed_list)
    test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, drop_last=False)
    
    all_predictions = []
    all_true_labels = []
    
    with torch.no_grad():
        for batch in test_dataloader:
            inputs, labels = batch
            
            # Initialize hidden states
            h_0 = torch.zeros(6, inputs.shape[0], model_classifier.hidden_dim).to(inputs)
            c_0 = torch.zeros(6, inputs.shape[0], model_classifier.hidden_dim).to(inputs)
            h = (h_0, c_0)
            
            # Get predictions
            outputs, _ = model_classifier.lstm_model(inputs, h)
            predictions = torch.argmax(outputs, dim=1)
            
            all_predictions.extend(predictions.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())
    
    return np.array(all_predictions), np.array(all_true_labels), model_classifier

# Use the test subjects from the current fold
test_subjects = asf[testing_index]
# best_model_path = model_path
best_model_path = r"D:\Sleep_Staging\W9S3_seResnet18_3Lstm.ckpt"
predicted_stages, original_stages, model_classifier = load_model_and_evaluate(best_model_path, test_subjects)

print(f"Predicted sleep stage sequence shape: {predicted_stages.shape}")
print(f"Original sleep stage sequence shape: {original_stages.shape}")
print(f"Test accuracy: {accuracy_score(original_stages, predicted_stages):.4f}")
print(f"Test F1-score: {f1_score(original_stages, predicted_stages, average='macro'):.4f}")

In [56]:
# Create model instances for downstream processing
modelCNN = resnet18(cnn_layers=CNN_LAYERS, in_lead=1)
model = SSModel(EMB_DIM, HIDDEN_DIM, NUM_LAYERS, NUM_CLASS, modelCNN, DROPOUT)
model = model.to('cuda')
checkpoint = torch.load(r"D:\Sleep_Staging\W9S3_seResnet18_3Lstm.ckpt")
new_checkpoint = {k.replace('lstm_model.', ''): v for k, v in checkpoint['state_dict'].items()}
model.load_state_dict(new_checkpoint)

<All keys matched successfully>

In [57]:
def Resample(input_signal, src_fs, tar_fs):
    if src_fs != tar_fs:
        dtype = input_signal.dtype
        audio_len = input_signal.shape[1]
        audio_time_max = 1.0 * (audio_len) / src_fs
        src_time = 1.0 * np.linspace(0, audio_len, audio_len) / src_fs
        tar_time = 1.0 * np.linspace(0, np.int64(audio_time_max * tar_fs), np.int64(audio_time_max * tar_fs)) / tar_fs
        for i in range(input_signal.shape[0]):
            if i == 0:
                output_signal = np.interp(tar_time, src_time, input_signal[i, :]).astype(dtype)
                output_signal = output_signal.reshape(1, len(output_signal))
            else:
                tmp = np.interp(tar_time, src_time, input_signal[i, :]).astype(dtype)
                tmp = tmp.reshape(1, len(tmp))
                output_signal = np.vstack((output_signal, tmp))
    else:
        output_signal = input_signal
    return output_signal

In [58]:
import torch.nn.functional as F
from torch.autograd import grad

class Grad_CAM():
    def __init__(self, model):
        self.model = model
        self.activations = None
        self.gradient = None
        
        # register hooks to capture the feature_map gradients
        def forward_hook(model, input, output):
            self.activations = output[0]
            
        def backward_hook(model, grad_input, grad_output):
            self.gradient = grad_output[0][0]
            
        feat_map = model.cnn.layer4[1].relu2
        self.forward_handle = feat_map.register_forward_hook(forward_hook)
        self.backward_handle = feat_map.register_backward_hook(backward_hook)
        
    def remove_hooks(self):
        """Remove hooks to prevent memory leaks"""
        if hasattr(self, 'forward_handle'):
            self.forward_handle.remove()
        if hasattr(self, 'backward_handle'):
            self.backward_handle.remove()
        
    def get_grad_cam(self, img, hid, indices=None, use_guided_backprop=False):
        self.model.eval()
        
        # Clear any existing gradients
        self.model.zero_grad()
        
        # Forward pass
        out = self.model(img, hid)
        
        num_features = self.activations.size()[0]
        topk = 5
        
        if indices is None:        
            values, indices = torch.topk(out[0], topk)
            values = values.detach().cpu()
            indices = indices.detach().cpu()
        
        # Compute heatmaps for each class
        heatmaps = torch.zeros(topk, 94)
        
        for i, c in enumerate(indices[0]):
            # Clear gradients before each backward pass
            self.model.zero_grad()
            
            # Get the raw logits (before softmax/log_softmax)
            # Use the raw output score instead of probability
            target_score = out[0][0][c]
            
            # Backward pass
            target_score.backward(retain_graph=True)
            
            if self.gradient is not None:
                # Feature importance using global average pooling
                feature_importance = self.gradient.mean(dim=[1]).detach().cpu()
                
                # Generate heatmap
                activations_cpu = self.activations.detach().cpu()
                
                heatmap = torch.zeros(94)
                for f in range(num_features):
                    heatmap += feature_importance[f] * activations_cpu[f]
                
                # Apply ReLU and normalize
                heatmap = F.relu(heatmap)
                
                # Improved normalization to handle small values
                if torch.max(torch.abs(heatmap)) > 1e-8:  # Use absolute max for better normalization
                    heatmap = heatmap / torch.max(torch.abs(heatmap))
                else:
                    print(f"Warning: Very small gradients for class {c}")
                
                heatmaps[i] = heatmap
            else:
                print(f"Warning: No gradients captured for class {c}")
                
        # Upsample to target size
        large_heatmaps = Resample(np.array(heatmaps), 94, 3000)  # Fixed: 94 instead of 100
        
        return large_heatmaps, values.data.numpy()[0] if indices is None else None, indices.data.numpy()[0]

    def __del__(self):
        """Cleanup hooks when object is destroyed"""
        self.remove_hooks()

In [59]:
def get_model_predictions(model, data, hidden_states, top_k=5):
    """
    Get model predictions separately from GradCAM computation
    
    Args:
        model: The trained model
        data: Input data tensor
        hidden_states: LSTM hidden states
        top_k: Number of top predictions to return
    
    Returns:
        dict containing predictions, probabilities, and indices
    """
    model.eval()
    
    with torch.no_grad():
        output, _ = model(data, hidden_states)
        probabilities = torch.exp(output[0])  # Convert log probabilities to probabilities
        top_probs, top_indices = torch.topk(probabilities, top_k)
        
        # Get prediction for the most likely class
        predicted_class = top_indices[0].item()
        predicted_prob = top_probs[0].item()
        
    return {
        'predicted_class': predicted_class,
        'predicted_probability': predicted_prob,
        'top_k_classes': top_indices.detach().cpu().numpy(),
        'top_k_probabilities': top_probs.detach().cpu().numpy(),
        'all_probabilities': probabilities.detach().cpu().numpy()
    }


def get_class_specific_gradcam(model, data, hidden_states, target_class=None, heatmap_size=94, upscale_to=3000):
    """
    Get GradCAM for a specific class or the predicted class
    
    Args:
        model: The trained model
        data: Input data tensor
        hidden_states: LSTM hidden states
        target_class: Specific class to generate GradCAM for (if None, uses predicted class)
        heatmap_size: Size of the feature map (default 94)
        upscale_to: Target size for upscaling heatmap (default 3000)
    
    Returns:
        dict containing heatmap and related information
    """
    # Set up hooks for GradCAM
    activations = None
    gradients = None
    
    def forward_hook(module, input, output):
        nonlocal activations
        activations = output[0]
        
    def backward_hook(module, grad_input, grad_output):
        nonlocal gradients
        gradients = grad_output[0][0]
    
    # Register hooks
    feat_map = model.cnn.layer4[1].relu2
    forward_handle = feat_map.register_forward_hook(forward_hook)
    backward_handle = feat_map.register_backward_hook(backward_hook)
    
    try:
        model.eval()
        model.zero_grad()
        
        # Forward pass
        output, _ = model(data, hidden_states)
        
        # If no target class specified, use the predicted class
        if target_class is None:
            target_class = torch.argmax(output[0]).item()
        
        # Backward pass for the target class
        if output[0].dim() == 1:
            # If output is 1D (single sample), index directly
            output[0][target_class].backward()
        else:
            # If output is 2D (batch dimension), index the first sample
            output[0][0][target_class].backward()
        
        # Compute GradCAM
        if gradients is not None and activations is not None:
            # Calculate feature importance (global average pooling of gradients)
            feature_importance = gradients.mean(dim=1).detach().cpu()
            activations_cpu = activations.detach().cpu()
            
            # Generate heatmap
            heatmap = torch.zeros(heatmap_size)
            num_features = activations_cpu.size(0)
            
            for f in range(num_features):
                heatmap += feature_importance[f] * activations_cpu[f]
            
            # Apply ReLU and normalize
            heatmap = F.relu(heatmap)
            if torch.max(heatmap) > 0:
                heatmap = heatmap / torch.max(heatmap)
            
            # Upscale heatmap
            if upscale_to != heatmap_size:
                heatmap_upscaled = Resample(heatmap.unsqueeze(0).numpy(), heatmap_size, upscale_to)[0]
            else:
                heatmap_upscaled = heatmap.numpy()
        else:
            print("Warning: Could not compute gradients or activations")
            heatmap_upscaled = np.zeros(upscale_to)
            
    finally:
        # Remove hooks
        forward_handle.remove()
        backward_handle.remove()
    
    return {
        'heatmap': heatmap_upscaled,
        'target_class': target_class,
        'feature_importance': feature_importance.numpy() if gradients is not None else None
    }


def get_all_class_gradcams(model, data, hidden_states, num_classes=5):
    """
    Get GradCAM for all classes
    
    Args:
        model: The trained model
        data: Input data tensor
        hidden_states: LSTM hidden states
        num_classes: Number of classes to generate GradCAMs for
    
    Returns:
        dict containing heatmaps for all classes
    """
    all_heatmaps = {}
    
    for class_idx in range(num_classes):
        gradcam_result = get_class_specific_gradcam(model, data, hidden_states, target_class=class_idx)
        all_heatmaps[f'class_{class_idx}'] = gradcam_result
    
    return all_heatmaps


def analyze_sample_with_gradcam(model, signal_data, channel_idx=2, class_names=None):
    """
    Complete analysis of a sample including predictions and GradCAM
    
    Args:
        model: The trained model
        signal_data: Loaded signal data (npz file)
        channel_idx: Channel index to use (default 2)
        class_names: List of class names for display
    
    Returns:
        dict containing all analysis results
    """
    if class_names is None:
        class_names = ["Wake", "N1", "N2", "N3", "REM"]
    
    # Prepare data
    data = torch.from_numpy(signal_data["x"][:,:,channel_idx].reshape(1, WINDOW, 3000)).float().to("cuda")
    hidden_states = (torch.zeros(6,1, 100).float().to("cuda"), torch.zeros(6,1, 100).float().to("cuda"))
    
    # Get predictions
    predictions = get_model_predictions(model, data, hidden_states)
    
    # Get GradCAM for predicted class
    predicted_gradcam = get_class_specific_gradcam(model, data, hidden_states, 
                                                  target_class=predictions['predicted_class'])
    
    # Get GradCAM for all classes
    all_gradcams = get_all_class_gradcams(model, data, hidden_states)
    
    # Get ground truth if available
    ground_truth = None
    if 'y' in signal_data:
        ground_truth = signal_data["y"][math.ceil(signal_data['y'].shape[0]/2)]
    
    return {
        'predictions': predictions,
        'predicted_gradcam': predicted_gradcam,
        'all_gradcams': all_gradcams,
        'ground_truth': ground_truth,
        'signal': signal_data["x"][:,:,channel_idx],
        'class_names': class_names
    }

# GradCAM

In [60]:
def visualize_predictions_and_gradcam(analysis_result, figsize=(30, 4), show_all_classes=True):
    """
    Visualize predictions and GradCAM results
    
    Args:
        analysis_result: Result from analyze_sample_with_gradcam function
        figsize: Figure size for plots
        show_all_classes: Whether to show GradCAM for all classes or just predicted
    """
    predictions = analysis_result['predictions']
    signal = analysis_result['signal']
    class_names = analysis_result['class_names']
    ground_truth = analysis_result['ground_truth']
    
    print("=== PREDICTION RESULTS ===")
    print(f"Predicted Class: {predictions['predicted_class']} ({class_names[predictions['predicted_class']]})")
    print(f"Predicted Probability: {predictions['predicted_probability']:.4f}")
    if ground_truth is not None:
        print(f"Ground Truth: {ground_truth} ({class_names[int(ground_truth)]})")
        print(f"Prediction Correct: {predictions['predicted_class'] == int(ground_truth)}")
    
    print(f"\nTop {len(predictions['top_k_classes'])} Predictions:")
    for i, (class_idx, prob) in enumerate(zip(predictions['top_k_classes'], predictions['top_k_probabilities'])):
        print(f"  {i+1}. {class_names[class_idx]}: {prob:.4f}")
    
    print("\n=== GRADCAM VISUALIZATION ===")
    
    if show_all_classes:
        # Show GradCAM for all classes
        all_gradcams = analysis_result['all_gradcams']
        
        for class_idx in range(len(class_names)):
            plt.figure(figsize=figsize)
            
            gradcam_data = all_gradcams[f'class_{class_idx}']
            heatmap = gradcam_data['heatmap']
            
            # Create heatmap overlay
            heatmap_overlay = np.vstack([heatmap] * 5)  # Stack for better visualization
            
            plt.imshow(np.expand_dims(heatmap_overlay, axis=2), cmap='Reds', 
                      aspect="auto", interpolation='nearest',
                      extent=[0, 3000, signal.min(), signal.max()], alpha=0.6)
            
            # Plot signal
            plt.plot(signal[2,:], 'k', linewidth=1, label='EEG Signal')
            
            title = f'Class {class_idx} ({class_names[class_idx]}) GradCAM'
            if class_idx == predictions['predicted_class']:
                title += ' [PREDICTED]'
            if ground_truth is not None and class_idx == int(ground_truth):
                title += ' [GROUND TRUTH]'
                
            plt.title(title, fontsize=16)
            plt.xlabel('Time Points', fontsize=12)
            plt.ylabel('Amplitude', fontsize=12)
            plt.colorbar(label='Activation Intensity')
            plt.legend()
            plt.tight_layout()
            plt.show()
    else:
        # Show only predicted class GradCAM
        plt.figure(figsize=figsize)
        
        predicted_gradcam = analysis_result['predicted_gradcam']
        heatmap = predicted_gradcam['heatmap']
        
        # Create heatmap overlay
        heatmap_overlay = np.vstack([heatmap] * 5)
        
        plt.imshow(np.expand_dims(heatmap_overlay, axis=2), cmap='Reds', 
                  aspect="auto", interpolation='nearest',
                  extent=[0, 3000, signal.min(), signal.max()], alpha=0.6)
        
        # Plot signal
        plt.plot(signal[2,:], 'k', linewidth=1, label='EEG Signal')
        
        predicted_class = predictions['predicted_class']
        title = f'Predicted Class {predicted_class} ({class_names[predicted_class]}) GradCAM'
        
        plt.title(title, fontsize=16)
        plt.xlabel('Time Points', fontsize=12)
        plt.ylabel('Amplitude', fontsize=12)
        plt.colorbar(label='Activation Intensity')
        plt.legend()
        plt.tight_layout()
        plt.show()


def batch_analyze_files(model, file_list, channel_idx=2, class_names=None):
    """
    Analyze multiple files and return predictions and ground truth
    
    Args:
        model: The trained model
        file_list: List of file paths to analyze
        channel_idx: Channel index to use
        class_names: List of class names
    
    Returns:
        dict containing predictions and ground truth for all files
    """
    if class_names is None:
        class_names = ["Wake", "N1", "N2", "N3", "REM"]
    
    all_predictions = []
    all_ground_truth = []
    all_probabilities = []
    
    print(f"Analyzing {len(file_list)} files...")
    
    for i, file_path in enumerate(tqdm(file_list)):
        try:
            signal_data = np.load(file_path)
            
            # Get only predictions (no GradCAM for efficiency)
            data = torch.from_numpy(signal_data["x"][:,:,channel_idx].reshape(1, WINDOW, 3000)).float().to("cuda")
            hidden_states = (torch.zeros(6,1, 100).float().to("cuda"), torch.zeros(6,1, 100).float().to("cuda"))
            
            predictions = get_model_predictions(model, data, hidden_states)
            
            all_predictions.append(predictions['predicted_class'])
            all_probabilities.append(predictions['predicted_probability'])
            
            # Get ground truth
            if 'y' in signal_data:
                ground_truth = signal_data["y"][math.ceil(signal_data['y'].shape[0]/2)]
                all_ground_truth.append(int(ground_truth))
            else:
                all_ground_truth.append(-1)  # Missing ground truth
                
        except Exception as e:
            print(f"Error processing file {file_path}: {e}")
            all_predictions.append(-1)
            all_ground_truth.append(-1)
            all_probabilities.append(0.0)
    
    return {
        'predictions': np.array(all_predictions),
        'ground_truth': np.array(all_ground_truth),
        'probabilities': np.array(all_probabilities),
        'class_names': class_names
    }

In [66]:
# Extract GradCAM visualizations for all epochs of patient SN_96
patient_id = 'SN_96'
patient_folder = os.path.join(base_folder, patient_id)

# Get all files for patient SN_96
patient_files = []
for file in os.listdir(patient_folder):
    if file.endswith('.npz'):
        patient_files.append(os.path.join(patient_folder, file))
patient_files.sort()

print(f"Processing {len(patient_files)} epochs for patient {patient_id}")

# Create output directory for GradCAM visualizations
gradcam_output_dir = os.path.join('gradcam_visualizations', patient_id)
os.makedirs(gradcam_output_dir, exist_ok=True)

# Process each epoch
for file_path in tqdm(patient_files, desc=f"Generating GradCAM for {patient_id}"):
    try:
        # Load signal data
        signal_data = np.load(file_path)
        file_name = os.path.basename(file_path).replace('.npz', '')
        
        # Prepare data
        signal = signal_data["x"][:, :, 2]  # Channel 2 (EEG)
        data_tensor = torch.from_numpy(signal.reshape(1, WINDOW, 3000)).float().to("cuda")
        hidden_states = (torch.zeros(6, 1, 100).float().to("cuda"), 
                        torch.zeros(6, 1, 100).float().to("cuda"))
        
        # Get prediction
        predictions = get_model_predictions(model, data_tensor, hidden_states)
        predicted_class = predictions['predicted_class']
        predicted_prob = predictions['predicted_probability']
        
        # Get ground truth
        ground_truth = None
        is_correct = False
        if 'y' in signal_data:
            ground_truth = int(signal_data["y"][math.ceil(signal_data['y'].shape[0]/2)])
            is_correct = (predicted_class == ground_truth)
        
        # Generate GradCAM for predicted class
        gradcam_result = get_class_specific_gradcam(model, data_tensor, hidden_states, 
                                                    target_class=predicted_class)
        
        # Visualize
        class_names = ["Wake", "N1", "N2", "N3", "REM"]
        middle_idx = signal.shape[0] // 2
        middle_signal = signal[middle_idx, :]
        heatmap = gradcam_result['heatmap']
        
        # Create figure
        fig, ax = plt.subplots(figsize=(20, 4))
        
        # Create heatmap overlay
        heatmap_overlay = np.vstack([heatmap] * 5)
        
        ax.imshow(np.expand_dims(heatmap_overlay, axis=2), cmap='Reds', 
                 aspect="auto", interpolation='nearest',
                 extent=[0, 3000, middle_signal.min(), middle_signal.max()], alpha=0.6)
        
        # Plot signal
        ax.plot(middle_signal, 'k', linewidth=1, label='EEG Signal')
        
        # Create title with prediction info
        status = "CORRECT" if is_correct else "WRONG"
        if ground_truth is not None:
            title = f'Prediction: {class_names[predicted_class]} ({predicted_prob:.3f}) | Truth: {class_names[ground_truth]} | Status: {status}'
        else:
            title = f'Prediction: {class_names[predicted_class]} ({predicted_prob:.3f})'
        
        ax.set_title(title, fontsize=24, fontweight='bold')
        ax.tick_params(axis='y', labelsize=18)
        ax.tick_params(axis='x', labelsize=18)
        ax.set_xlabel('Time Points', fontsize=20)
        ax.set_ylabel('Amplitude', fontsize=20)
        # ax.legend()
        # plt.tight_layout()
        
        # Save figure
        save_path = os.path.join(gradcam_output_dir, f'{file_name}_gradcam.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        continue

print(f"\nGradCAM visualizations saved to {gradcam_output_dir}")
print(f"Total epochs processed: {len(patient_files)}")

Processing 208 epochs for patient SN_96


Generating GradCAM for SN_96: 100%|██████████| 208/208 [01:17<00:00,  2.70it/s]


GradCAM visualizations saved to gradcam_visualizations\SN_96
Total epochs processed: 208
